# Multi-Object Detection and Tracking (Demo Notebook)
This notebook demonstrates the pipeline using YOLOv8 and ByteTrack.

In [ ]:
!pip install ultralytics supervision opencv-python

In [ ]:
import cv2
from ultralytics import YOLO
import supervision as sv
import numpy as np

In [ ]:
# 1. Load Model
model = YOLO('yolov8m.pt')

# 2. Initialize Tracker
tracker = sv.ByteTrack()

In [ ]:
SOURCE_VIDEO_PATH = 'input_video.mp4'
TARGET_VIDEO_PATH = 'output_video.mp4'
video_info = sv.VideoInfo.from_video_path(video_path=SOURCE_VIDEO_PATH)

box_annotator = sv.BoundingBoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=2, text_scale=0.5, text_padding=10)
trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=60)
heatmap_annotator = sv.HeatMapAnnotator()

In [ ]:
def process_frame(frame: np.ndarray, index: int) -> np.ndarray:
    results = model(frame, verbose=False, classes=[0])[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = tracker.update_with_detections(detections)

    labels = [f'#{tracker_id} {confidence:0.2f}' for confidence, tracker_id in zip(detections.confidence, detections.tracker_id)]

    annotated_frame = frame.copy()
    annotated_frame = heatmap_annotator.annotate(scene=annotated_frame, detections=detections)
    annotated_frame = trace_annotator.annotate(scene=annotated_frame, detections=detections)
    annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

    count_text = f'Live Object Count: {len(detections)}'
    cv2.putText(annotated_frame, count_text, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA)

    return annotated_frame

In [ ]:
print('Processing video...')
sv.process_video(source_path=SOURCE_VIDEO_PATH, target_path=TARGET_VIDEO_PATH, callback=process_frame)
print('Finished processing!')